# PGM E0.R R2 — registered zero-fit aggregate
Consumes the frozen R1 substrate and the completed checkpoint shards A–D. Requires actual plus all geometry seeds 42–61, executes no model fit, does not access PrivateTest, and stops before M0.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys
REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-motif-e0r'
ORCHESTRATION_SHA = '8b43d930c62b5f4fe808fd596645bff7e10dcf65'
PROJECT = Path('/kaggle/working/FER2013_Graph_E0R_AGGREGATE')
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',ORCHESTRATION_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != ORCHESTRATION_SHA: raise RuntimeError(f'orchestration source lock mismatch: {head}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
slugs = ('pgm-e0r-r2-shard-a-v539','pgm-e0r-r2-shard-b-v2','pgm-e0r-r2-shard-c-v3','pgm-e0r-r2-shard-d-v3')
sentinels = ('e0r_r2_actual.npz','e0r_r2_seed_46.npz','e0r_r2_seed_51.npz','e0r_r2_seed_56.npz')
roots = []
for slug, sentinel in zip(slugs, sentinels):
    candidates = (Path('/kaggle/input') / slug, Path('/kaggle/input/datasets/irthn1311') / slug)
    matches = [root for root in candidates if (root / sentinel).is_file()]
    if len(matches) != 1: raise RuntimeError(f'need exactly one mount for {slug}; matches={matches}')
    roots.append(matches[0])
entry = PROJECT / 'notebooks/e0r-r2-aggregate-entry.py'
command = [sys.executable, str(entry)]
for root in roots: command.extend(['--checkpoint-root', str(root)])
subprocess.run(command, check=True)
